In [1]:
import os  
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = 'telecom_guide.pdf'

loader = PyPDFLoader(PDF_PATH)

#splits the doc. page by page
pages = loader.load()

print(f"Loaded {len(pages)} pages from the PDF.")
print("\n--- First page preview (first 300 chars) ---")
print(pages[1].page_content[:300])
print(pages[2].metadata)

C:\Users\chath\AppData\Local\Temp\ipykernel_27172\4166839181.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
y:\Projects\AgenticAI\AgenticAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 9 pages from the PDF.

--- First page preview (first 300 chars) ---
Telecom Technical Reference Guide  - Internal Use Only
1. Introduction to Mobile Networks
Mobile networks have evolved through several generations, each offering significant improvements in speed,
capacity, and capability.
2G (GSM) networks introduced digital voice and basic data services such as SM
{'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-04-30T14:16:24+00:00', 'source': 'telecom_guide.pdf', 'total_pages': 9, 'page': 2, 'page_label': '3'}


In [3]:
### chunking ###

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600, #600 characters
    chunk_overlap = 100, #overlap keeps context at boundaries
    separators = ["\n\n", "\n", ".", " "] #tries paragraph > line > sentence > word
)

chunks = splitter.split_documents(pages)
print(f"Created {len(chunks)} chunks.")

Created 37 chunks.


In [4]:
print(chunks[0].page_content)

Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    encode_kwargs={"normalize_embeddings": True},
)
vector_store = Chroma.from_documents(chunks, embeddings)

print(f"Vector store ready. {vector_store._collection.count()} vectors stored.")

y:\Projects\AgenticAI\AgenticAI\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\chath\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
y:\Projects\AgenticAI\AgenticAI\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system us

Vector store ready. 37 vectors stored.


In [6]:
retriever = vector_store.as_retriever(
    search_type="mmr", search_kwargs={"k": 3}  #top 3 similar chunks
)

test_query = "What is VoLTE and how does it improve call quality?"
retrieved = retriever.invoke(test_query)

for i, doc in enumerate(retrieved, 1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:300])
    print()

--- Chunk 1 ---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

--- Chunk 2 ---
prioritised over general data traffic. This prevents voice quality degradation during periods of network congestion.
Without QoS, voice packets would compete with video streaming and file downloads, causing jitter and packet
loss.
Fallback Behaviour: If a VoLTE call cannot be established  - for exam

--- Chunk 3 ---
curiosity and is connected to a premium-rate number. Educating customers not to call back unknown international
numbers reduces exposure.
Account Takeover: Attackers use credentials obtained from data breaches to log in to the MyTelecom app and
make unauthorised changes (plan downgrades, SIM swaps, 



In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# --- Helper: join retrieved chunks into a single context string ---
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)


# --- System prompt: ground the LLM in the retrieved context ---
SYSTEM_PROMPT = """\
You are a helpful telecom assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

llm = ChatGoogleGenerativeAI(
    model="gemma-4-31b-it",
    temperature=1.0,  # Gemini 3.0+ defaults to 1.0
)

# Chain: prompt1 → llm → parser → prompt2 → llm → parser
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


# Invoke
result = chain.invoke({"country": "NewZealand"})
print(result)

NameError: name 'format_docs' is not defined